# Prepare GSE68086 for cancer-type classification
This notebook selects five cancer classes, transposes the gene-expression matrix so samples are rows, and writes a Galaxy tabular output.

In [ ]:
import csv
from collections import Counter
import pandas as pd

series_file = 'galaxy_input/series/GSE68086_series_matrix.txt'
expression_file = 'galaxy_input/gene_expression/GSE68086_TEP_data_matrix.txt'
wanted_cancers = {'Breast', 'CRC', 'Lung', 'GBM', 'Pancreas'}

sample_ids = []
cancer_types = []
with open(series_file, newline='') as handle:
    for row in csv.reader(handle, delimiter='\t'):
        if not row:
            continue
        values = [value.strip('\"') for value in row[1:]]
        if row[0] == '!Sample_geo_accession':
            sample_ids = values
            cancer_types = [''] * len(sample_ids)
        elif row[0] == '!Sample_characteristics_ch1':
            for i, value in enumerate(values):
                if value.lower().startswith('cancer type:'):
                    cancer_types[i] = value.split(':', 1)[1].strip()
print(Counter(cancer_types))

In [ ]:
expression = pd.read_csv(expression_file, sep='\t', compression='infer')
gene_column = expression.columns[0]
expression_columns = expression.columns[1:]
assert len(expression_columns) == len(cancer_types)
column_mapping = [
    {'column_name': column, 'sample_id': sample_id, 'cancer_type': cancer_type}
    for column, sample_id, cancer_type in zip(expression_columns, sample_ids, cancer_types)
    if cancer_type in wanted_cancers
]
print('Selected samples:', len(column_mapping))
print(Counter(item['cancer_type'] for item in column_mapping))

In [ ]:
selected_columns = [item['column_name'] for item in column_mapping]
filtered = expression[[gene_column] + selected_columns].copy()
transposed = filtered.set_index(gene_column).T
transposed.index = [item['cancer_type'] for item in column_mapping]
transposed.index.name = 'label'
transposed.reset_index(inplace=True)
transposed.to_csv('outputs/GSE68086_expression_columns.tabular', sep='\t', index=False)
print('Output shape:', transposed.shape)
print(transposed['label'].value_counts())
transposed.head()